In [2]:
import regex as re
import importlib
import tokenizer
importlib.reload(tokenizer)
st = tokenizer.SimpleTokenizer
import data_loader
importlib.reload(data_loader)
gt  = data_loader.GPTDataset
from torch.utils.data import DataLoader

In [3]:
with open("the-verdict.txt","r",encoding="utf-8") as f:
    raw_text = f.read()
print("Total number of characters:",len(raw_text))
print(raw_text[:99])

Total number of characters: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


In [4]:
preprocessed_text = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed_text = [token.split()[0] for token in preprocessed_text if token.strip()]
print("Total number of tokens:",len(preprocessed_text))
print(preprocessed_text[:30])

Total number of tokens: 4690
['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


In [5]:
ordered_list = sorted(set(preprocessed_text))
vocab_size = len(ordered_list)
print("Vocabulary size:",vocab_size)


Vocabulary size: 1130


In [6]:
vocab_dict = {token:idx for idx,token in enumerate(ordered_list)}
print("Vocabulary dictionary:",dict(list(vocab_dict.items())[:30]))

Vocabulary dictionary: {'!': 0, '"': 1, "'": 2, '(': 3, ')': 4, ',': 5, '--': 6, '.': 7, ':': 8, ';': 9, '?': 10, 'A': 11, 'Ah': 12, 'Among': 13, 'And': 14, 'Are': 15, 'Arrt': 16, 'As': 17, 'At': 18, 'Be': 19, 'Begin': 20, 'Burlington': 21, 'But': 22, 'By': 23, 'Carlo': 24, 'Chicago': 25, 'Claude': 26, 'Come': 27, 'Croft': 28, 'Destroyed': 29}


In [7]:
t = st(raw_text)
text = """"It's the last he painted, you know," Mrs. Gisburn said with pardonable pride."""
ids = t.encode(text)
print("Encoded ids:",ids)

Encoded ids: [1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]


In [8]:
decoded_text = t.decode(ids)
print("Decoded text:",decoded_text)

Decoded text: " It's the last he painted, you know," Mrs. Gisburn said with pardonable pride.


In [9]:
text = "Hello, world! It' s a test."
ids = t.encode(text)
print("Encoded ids:",ids)

Encoded ids: [1134, 5, 1134, 0, 56, 2, 850, 115, 1134, 7]


In [10]:
text1 = "Hello, world! It's a test."
text2 = "Here is such a big palace that it can be used to store a lot of things."
text = "<|BOS|> " + text1 + " <|EOS|> " + text2 + " <|endoftext|>"
ids = t.encode(text)
print("Encoded ids:",ids)

Encoded ids: [1131, 1134, 5, 1134, 0, 56, 2, 850, 115, 1134, 7, 1133, 1134, 584, 949, 115, 219, 1134, 987, 585, 244, 198, 1057, 1016, 1134, 115, 1134, 722, 997, 7, 1130]


In [11]:
import tiktoken
print("tiktoken version:", importlib.metadata.version("tiktoken"))


tiktoken version: 0.13.0


In [12]:
tokenizer = tiktoken.get_encoding("gpt2")

In [13]:
text1 = "Hello, world! It's a test."
text2 = "Here is such a big palace that it can be used to store alotofthings."
text = " <|endoftext|> ".join([text1, text2])
ids = tokenizer.encode(text,allowed_special={"<|endoftext|>"})
print("Encoded ids:",ids)

Encoded ids: [15496, 11, 995, 0, 632, 338, 257, 1332, 13, 220, 50256, 3423, 318, 884, 257, 1263, 20562, 326, 340, 460, 307, 973, 284, 3650, 43158, 1659, 27971, 13]


In [14]:
text_decoded = tokenizer.decode(ids)
print("Decoded text:",text_decoded)

Decoded text: Hello, world! It's a test. <|endoftext|> Here is such a big palace that it can be used to store alotofthings.


In [15]:
enc_text = tokenizer.encode(raw_text)
len_enc_text = len(enc_text)
print("Length of encoded text:",len_enc_text)

Length of encoded text: 5145


In [16]:
enc_sample = enc_text[50:]

In [17]:
context_size = 4
input_array = enc_sample[:context_size]
output_array = enc_sample[1:context_size+1]
print("Input array:",input_array)
print("Output array:",output_array)

Input array: [290, 4920, 2241, 287]
Output array: [4920, 2241, 287, 257]


In [18]:
for i in range(context_size):
    context = input_array[:i+1]
    output = output_array[i]
    print(f"Context: {context}, Output: {output}")

Context: [290], Output: 4920
Context: [290, 4920], Output: 2241
Context: [290, 4920, 2241], Output: 287
Context: [290, 4920, 2241, 287], Output: 257


In [19]:
for i in range(context_size):
    context = enc_sample[:i+1]
    output = enc_sample[i+1]
    print(f"Context: {tokenizer.decode(context)}, Output: {tokenizer.decode([output])}")

Context:  and, Output:  established
Context:  and established, Output:  himself
Context:  and established himself, Output:  in
Context:  and established himself in, Output:  a


In [20]:
def create_data_loader(txt, batch_size=4, context_size=256,
                       stride=128,shuffle=True,drop_last=True,
                       cpu_thread_number=4):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = gt(txt, tokenizer, context_size, stride)
    data_loader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle,
                             drop_last=drop_last, num_workers=cpu_thread_number)
    return data_loader


In [21]:
import torch
print("PyTorch version:", torch.__version__)

PyTorch version: 2.11.0+cu128


In [ ]:
dataloader = create_data_loader(raw_text, batch_size=10, context_size=4, stride=1,
                                shuffle=False, drop_last=True, cpu_thread_number=4)
data_iter = iter(dataloader)
first_batch = next(data_iter)

First batch: tensor([[   40,   367,  2885,  1464],
        [  367,  2885,  1464,  1807],
        [ 2885,  1464,  1807,  3619],
        [ 1464,  1807,  3619,   402],
        [ 1807,  3619,   402,   271],
        [ 3619,   402,   271, 10899],
        [  402,   271, 10899,  2138],
        [  271, 10899,  2138,   257],
        [10899,  2138,   257,  7026],
        [ 2138,   257,  7026, 15632]])


In [ ]:
print("First batch Input:", first_batch[0])
print("First batch Target:", first_batch[1])